In [1]:
!pip install umap-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

datasets = {
    "full": "../../data/processed/ProcessedData.csv",
    "pca":  "../../data/reduced/dataPCA.csv",
    "fs":   "../../data/reduced/dataFS.csv"
}

outDir = "../../results/kmeans"

In [19]:
import umap

In [20]:
def elbowMethod(X, name):
    distortions = []
    ks = range(2, 15)

    for k in ks:
        km = KMeans(n_clusters=k, n_init=20, random_state=42)
        km.fit(X)
        distortions.append(km.inertia_)

    plt.figure(figsize=(6,4))
    plt.plot(ks, distortions, marker="o")
    plt.xlabel("K")
    plt.ylabel("Inercia")
    plt.title(f"Elbow Metod ({name})")
    plt.grid()
    plt.tight_layout()
    plt.savefig(f"{outDir}/elbow_{name}.png", dpi=200)
    plt.close()

    return ks, distortions

In [21]:
results = []
for name, path in datasets.items():
    
    df = pd.read_csv(path)
    
    if "Class" in df.columns:
        y1 = df["Class"].values
        X = df.drop(columns=["Class"])
    else:
        y1 = None
        X = df

    if "V129" in X.columns:
        y2 = X["V129"].values
        X = X.drop(columns=["V129"])
    else:
        y2 = None

    ks, distortions = elbowMethod(X, name)
    diff2 = np.diff(distortions, 2)
    k = ks[np.argmax(diff2) + 2]

    kmeans = KMeans(
        n_clusters=k,
        n_init=20,
        random_state=42
    )

    labels = kmeans.fit_predict(X)

    sil = silhouette_score(X, labels)
    db  = davies_bouldin_score(X, labels)

    if y1 is not None:
        ari = adjusted_rand_score(y1, labels)
        nmi = normalized_mutual_info_score(y1, labels)
    else:
        ari, nmi = None, None

    results.append({
        "Dataset": name,
        "Algorithm": "KMeans",
        "K": k,
        "Silhouette": sil,
        "Davies_Bouldin": db,
        "ARI": ari,
        "NMI": nmi
    })

    clusters = X.copy()
    clusters["cluster"] = labels
    clusters.to_csv(
        f"{outDir}/kmeans_{name}_clusters.csv",
        index=False
    )

    reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = reducer.fit_transform(X)

    plt.figure(figsize=(6,5))
    plt.scatter(X_umap[:,0], X_umap[:,1], c=labels, s=5, cmap="tab10")
    plt.title(f"KMeans UMAP ({name})")
    plt.tight_layout()
    plt.savefig(f"{outDir}/kmUMAP_{name}.png", dpi=200)
    plt.close()

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X)

    plt.figure(figsize=(6,5))
    plt.scatter(X_tsne[:,0], X_tsne[:,1], c=labels, s=5, cmap="tab10")
    plt.title(f"KMeans TSNE ({name})")
    plt.tight_layout()
    plt.savefig(f"{outDir}/kmTSNE_{name}.png", dpi=200)
    plt.close()

C:\Users\Korisnik\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\Korisnik\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\Korisnik\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [22]:
summaryData = pd.DataFrame(results)
summaryData.to_csv(f"{outDir}/kmeans_summary.csv", index=False)
print(summaryData)

  Dataset Algorithm  K  Silhouette  Davies_Bouldin       ARI      NMI
0    full    KMeans  4    0.379531        1.251744  0.099159  0.22559
1     pca    KMeans  4    0.399006        1.185537       NaN      NaN
2      fs    KMeans  4    0.370770        1.163873       NaN      NaN
